In [0]:
from pyspark.sql import functions as F

In [0]:
def ingestion_timestamp(df_final):
    return (
        df_final
            .withColumn('created_timestamp', F.current_timestamp())
            .withColumn('updated_timestamp', F.current_timestamp())
    )

In [0]:
from delta.tables import DeltaTable

def write_to_silver(
    df,
    target_table,
    table_key,
    columns_to_update
):
    df = ingestion_timestamp(df)

    if not spark.catalog.tableExists(target_table):
        (
            df.write
                .format('delta')
                .mode('overwrite')
                .saveAsTable(target_table)
        )
    else:

        delta_table = DeltaTable.forName(spark, target_table)

        map_to_update = {column: f's.{column}' for column in columns_to_update}
        map_to_update['updated_timestamp'] = df['updated_timestamp']

        (
            delta_table.alias('t')
            .merge (
                df.alias('s'),
                table_key
            )
            .whenMatchedUpdate(
                condition='s.batch_id >= t.batch_id',
                set = map_to_update
            )
            .whenNotMatchedInsertAll()
        )

